# Vyuha P16 — head-to-head on PIArena (vs the cited attackers)

Slots **Vyuha** into **PIArena** (Geng et al., ACL 2026 — arXiv:2604.08499) as a Filter-type defense
and attacks it with the *same* static + search suite **PISmith** (Yin et al., COLM 2026 —
arXiv:2603.13026) benchmarks on: **Direct, Combined, Ignore, Completion, Character** (static) and
**PAIR / TAP** (search). Reports Vyuha's **ASR@1** next to the `none` baseline, so it can be placed
against PIArena's published defenses (PIGuard, PromptGuard, DataSentinel, …).

**Honest scope.** PISmith's *RL* attacker trains an attack LLM (GRPO, ~4 GPUs, no released
checkpoint) — out of free-compute scope; it is the cited upper bound. This notebook runs the
free-compute-feasible part: static attacks on a single GPU (+ light search).

**Requirements.** Kaggle/Colab **GPU** session, **Internet ON**, and a **HF token** (some Vyuha
training datasets are gated — accept their terms first). The adapter lives in the Vyuha repo at
`integrations/piarena/`; this notebook only orchestrates clone → install → register → run → collect.

## 1 · Setup — clone Vyuha + PIArena, install, register the `vyuha` defense

In [ ]:
import os, sys, glob, subprocess, shutil, textwrap

VYUHA_URL = "https://github.com/g25ait2149/vyuha.git"
PIARENA_URL = "https://github.com/sleeepeer/PIArena.git"
WORK = "/kaggle/working"
VYUHA_DIR = f"{WORK}/vyuha_src"
PIARENA_DIR = f"{WORK}/PIArena"

def sh(cmd, **kw):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, **kw)

# --- Vyuha (adapter + vyuha package + eval) ---
if os.path.isdir(f"{VYUHA_DIR}/.git"):
    sh(f'git -C "{VYUHA_DIR}" pull --ff-only')
else:
    sh(f'git clone --depth 1 {VYUHA_URL} "{VYUHA_DIR}"')
hits = glob.glob(VYUHA_DIR + "/**/vyuha/__init__.py", recursive=True)
VYUHA_ROOT = os.path.dirname(os.path.dirname(hits[0])) if hits else VYUHA_DIR
print("vyuha repo root:", VYUHA_ROOT)

# --- PIArena (the platform) ---
if os.path.isdir(f"{PIARENA_DIR}/.git"):
    sh(f'git -C "{PIARENA_DIR}" pull --ff-only')
else:
    sh(f'git clone --depth 1 {PIARENA_URL} "{PIARENA_DIR}"')

# --- install PIArena + deps (editable) ---
sh(f'cd "{PIARENA_DIR}" && pip -q install -r requirements.txt && pip -q install -e . && pip -q install --upgrade setuptools pip')

# --- drop the Vyuha adapter into piarena/defenses/ and register it ---
adapter_src = f"{VYUHA_ROOT}/integrations/piarena"
dest = f"{PIARENA_DIR}/piarena/defenses"
for f in ("_vyuha_core.py", "defense_vyuha.py"):
    shutil.copy(f"{adapter_src}/{f}", f"{dest}/{f}")
    print("copied", f, "->", dest)
init = f"{dest}/__init__.py"
reg = "from .defense_vyuha import VyuhaDefense  # noqa: F401"
txt = open(init).read()
if reg not in txt:
    open(init, "a").write("\n" + reg + "\n")
    print("registered VyuhaDefense in", init)
else:
    print("VyuhaDefense already registered")

# --- so PIArena's process can import `vyuha` and `eval` from the Vyuha repo ---
os.environ["PYTHONPATH"] = VYUHA_ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")
print("PYTHONPATH ->", os.environ["PYTHONPATH"][:120], "...")
print("\nSetup done.")

## 2 · Hugging Face login (gated datasets)
Vyuha's L1 detector is fit on assembled corpora, some **gated** (AdvBench, HarmBench, WildGuardMix).
Accept their terms on HF, then set your token below (or add a Kaggle Secret `HF_TOKEN`).

In [ ]:
import os
# Option A: paste (session-only). Option B: Kaggle Secret.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secret")
except Exception:
    os.environ.setdefault("HF_TOKEN", "")   # <-- or paste your token string here
    print("HF_TOKEN set:", bool(os.environ["HF_TOKEN"]))
os.environ.setdefault("HUGGINGFACE_HUB_TOKEN", os.environ.get("HF_TOKEN", ""))
if os.environ.get("HF_TOKEN"):
    subprocess.run(f'huggingface-cli login --token {os.environ["HF_TOKEN"]} --add-to-git-credential', shell=True)

## 3 · Config — dataset, attacks, defenses to compare

In [ ]:
DATASET = "squad_v2"                 # a PIArena dataset (see HF: sleeepeer/PIArena)
STATIC_ATTACKS = ["direct", "combined", "ignore", "completion", "character"]
SEARCH_ATTACKS = ["pair"]            # heavier; add "tap","strategy_search" if you have budget
DEFENSES = ["none", "vyuha"]         # baseline vs Vyuha (add "piguard","promptguard" to compare live)
BACKEND_LLM = "Qwen/Qwen3-4B-Instruct-2507"   # target; fits a single T4
ATTACKER_LLM = "Qwen/Qwen3-4B-Instruct-2507"  # for search attacks
RUN_SEARCH = False                   # set True to also run PAIR/TAP (slow)
NUM_SAMPLES = 5                      # pass@k for search attacks (keep small on free tier)

# Optional: turn on Vyuha's L2 guard (Qwen3Guard-0.6B). Requires a bit more VRAM.
VYUHA_USE_GUARD = False
import os, json
if VYUHA_USE_GUARD:
    os.environ["VYUHA_DEFENSE_CONFIG"] = json.dumps({"use_guard": True, "guard_preset": "qwen3guard"})
print("config set. dataset:", DATASET, "| defenses:", DEFENSES)

## 3b · Smoke test — is `vyuha` registered and does PIArena run?
Runs two checks with **full output printed inline** (no log files needed). If either fails, the error here tells us exactly what to fix before the batch.

In [ ]:
import os, subprocess
env = {**os.environ}
# (1) is the Vyuha defense importable + registered inside PIArena's env?
chk = subprocess.run(['python','-c',
    'from piarena.defenses import get_defense; d=get_defense("vyuha"); print("OK registered:", d)'],
    cwd=PIARENA_DIR, env=env, capture_output=True, text=True)
print('--- defense import check ---'); print(chk.stdout or ''); print(chk.stderr or ''); print('exit', chk.returncode)
# (2) does one minimal PIArena run work at all? (no attack, no defense) - full output
smoke = subprocess.run(['python','main.py','--dataset',DATASET,'--attack','none','--defense','none'],
    cwd=PIARENA_DIR, env=env, capture_output=True, text=True)
print('\n--- main.py smoke (attack=none, defense=none) ---')
print((smoke.stdout or '')[-3000:]); print('--- stderr ---'); print((smoke.stderr or '')[-3000:]); print('exit', smoke.returncode)


## 4 · Run the static attacks
Each run: `python main.py --dataset <d> --attack <a> --defense <def>`. We capture stdout and try to
parse an ASR value; the full log is saved to `/kaggle/working/piarena_logs/` regardless.

In [ ]:
import os, re, subprocess, itertools, pathlib, json
LOGDIR = '/kaggle/working/piarena_logs'; os.makedirs(LOGDIR, exist_ok=True)
env = {**os.environ}

def run_one(entry, attack, defense, search=False):
    script = 'main_search.py' if search else 'main.py'
    cmd = ['python', script, '--dataset', DATASET, '--attack', attack, '--defense', defense]
    if search:
        cmd += ['--backend_llm', BACKEND_LLM, '--attacker_llm', ATTACKER_LLM, '--num_samples', str(NUM_SAMPLES)]
    print('\n>>>', ' '.join(cmd))
    p = subprocess.run(cmd, cwd=PIARENA_DIR, env=env, capture_output=True, text=True)
    out = (p.stdout or '') + '\n' + (p.stderr or '')
    (pathlib.Path(LOGDIR)/f'{entry}.log').write_text(out)
    m = re.findall(r'ASR[^0-9]{0,12}(\d+\.\d+|\d+(?:\.\d+)?%?)', out, flags=re.I)
    asr = m[-1] if m else None
    print('   exit', p.returncode, '| parsed ASR:', asr)
    if p.returncode != 0 or asr is None:            # show what happened, inline, no file hunting
        print('   --- output tail (diagnose) ---')
        print('\n'.join(out.strip().splitlines()[-25:]))
    return {'attack': attack, 'defense': defense, 'search': search, 'exit': p.returncode, 'asr': asr}

results = []
for atk, dfn in itertools.product(STATIC_ATTACKS, DEFENSES):
    results.append(run_one(f'{atk}__{dfn}', atk, dfn, search=False))
print('\nstatic runs complete:', len(results))


## 5 · (Optional) search-based attacks — PAIR / TAP

In [ ]:
import itertools
if RUN_SEARCH:
    for atk, dfn in itertools.product(SEARCH_ATTACKS, DEFENSES):
        results.append(run_one(f"{atk}__{dfn}", atk, dfn, search=True))
    print("search runs complete.")
else:
    print("RUN_SEARCH=False — skipping PAIR/TAP. Set True in the config cell to include them.")

## 6 · Collect results — Vyuha vs baseline (+ any live defenses)

In [ ]:
import glob, json, pathlib, pandas as pd

# 1) primary: the parsed-stdout table
df = pd.DataFrame(results)
# 2) also surface any structured result files PIArena wrote (json/csv), for exact numbers
found = []
for pat in ("results/**/*.json", "results/**/*.csv", "outputs/**/*.json", "**/*results*.json"):
    found += glob.glob(f"{PIARENA_DIR}/{pat}", recursive=True)
found = sorted(set(found))
print("PIArena result files found:", len(found))
for f in found[:20]:
    print("  ", f.replace(PIARENA_DIR+"/", ""))

if not df.empty:
    piv = df[~df.search].pivot_table(index="attack", columns="defense", values="asr", aggfunc="first")
    print("\n=== ASR@1 (static) — parsed from stdout ===")
    print(piv.to_string())
    df.to_csv("/kaggle/working/piarena_vyuha_results.csv", index=False)
    print("\nsaved -> /kaggle/working/piarena_vyuha_results.csv")
print("\nIf 'asr' is None, open the matching log in /kaggle/working/piarena_logs/ and read the exact")
print("ASR line, or the structured result file above — then tighten the regex in cell 4. Report the")
print("real numbers whatever they are (including if Vyuha's ASR exceeds PIGuard/DataSentinel).")

## 7 · Interpretation & what to send back
- **Comparison to publish:** Vyuha's **ASR@1 per attack** vs `none`, placed next to PIArena's published
  defenses. In their paper (Qwen3-4B target) the filter defenses sit around: PIGuard 0.82, PromptGuard
  0.89, DataSentinel 0.52 (PISmith ASR@1). Static/search baselines are far weaker (Direct 0.04,
  Combined 0.07, PAIR 0.16, TAP 0.11) — that is the band this run measures Vyuha against.
- **Utility:** run `--attack none --defense vyuha` to get task accuracy without attack (the utility
  column); Vyuha passes benign contexts through unchanged, so utility should stay near the `none`
  baseline.
- **Honesty rule:** report the measured ASR as-is. If Vyuha underperforms a published filter, that is a
  finding for the results + Limitations, not something to bury.
- **Send me:** `piarena_vyuha_results.csv` (or the printed table) and I'll fold it into the paper's
  results table and §7.4, alongside the pairwise + genetic adaptive numbers from P14.

*The RL attacker (PISmith) itself needs a ~4-GPU training rig and is cited as the stronger upper bound;
this notebook covers the free-compute-feasible static + search head-to-head.*